In [1]:
# Step 1: Install Required Dependencies
!pip install -q transformers sentence-transformers faiss-cpu pypdf reportlab gradio langchain-text-splitters accelerate torch pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.9 MB/s eta 0:00:00:00:01


In [2]:
# Step 2: Imports & System Setup
import os
import re
import json
import gc
import warnings
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple

# PyPDF
from pypdf import PdfReader, PdfWriter

# FAISS & Sentence Transformers
import faiss
from sentence_transformers import SentenceTransformer

# Hugging Face Transformers & Torch
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# LangChain Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ReportLab PDF Generation
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, KeepTogether, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# Gradio
import gradio as gr

warnings.filterwarnings('ignore')

# Detect Hardware Accelerator
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('System Initialized.')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


System Initialized.
Device: cuda
GPU Name: Tesla T4
VRAM Available: 15.64 GB


In [3]:
# Step 3: PDF Extraction & Text Parsing Utilities

def extract_text_from_pdf(pdf_source) -> str:
    """Extracts raw text from a PDF file path or file object."""
    if pdf_source is None:
        return ''
    
    if isinstance(pdf_source, dict) and 'name' in pdf_source:
        file_path = pdf_source['name']
    elif hasattr(pdf_source, 'name'):
        file_path = pdf_source.name
    else:
        file_path = str(pdf_source)
        
    if not os.path.exists(file_path):
        return f'Error: File not found at {file_path}'
        
    try:
        reader = PdfReader(file_path)
        extracted_text = []
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                extracted_text.append(page_text)
        
        full_text = '\n'.join(extracted_text)
        return full_text.strip()
    except Exception as e:
        return f'Error extracting PDF text: {str(e)}'

def clean_text(text: str) -> str:
    """Cleans extracted text by normalizing whitespace."""
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

print('Text Extraction Utilities Loaded.')


Text Extraction Utilities Loaded.


In [4]:
# Step 4: Hugging Face Models Setup (LLM + Embeddings)

print('Loading Embedding Model (sentence-transformers/all-MiniLM-L6-v2)...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding Model Loaded Successfully.')

DEFAULT_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct' if DEVICE == 'cuda' else 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = None
llm_model = None
llm_pipeline = None

def load_llm_model(model_id: str = DEFAULT_MODEL_ID):
    global tokenizer, llm_model, llm_pipeline
    print(f'Loading Hugging Face LLM: {model_id} on {DEVICE}...')
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    
    if DEVICE == 'cuda':
        llm_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=torch.float16,
            device_map='auto'
        )
    else:
        llm_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=torch.float32
        )
        
    if hasattr(llm_model, 'generation_config') and llm_model.generation_config is not None:
        llm_model.generation_config.max_length = None
        llm_model.generation_config.max_new_tokens = None
        
    llm_pipeline = pipeline(
        'text-generation',
        model=llm_model,
        tokenizer=tokenizer,
        temperature=0.3,
        do_sample=True,
        repetition_penalty=1.1,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f'LLM Model {model_id} Loaded Ready.')

# Initial Load
load_llm_model(DEFAULT_MODEL_ID)

def generate_llm_response(system_prompt: str, user_prompt: str, max_tokens: int = 600) -> str:
    """Generates a response from HF LLM using chat message format."""
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ]
    try:
        outputs = llm_pipeline(messages, max_new_tokens=max_tokens)
        reply = outputs[0]['generated_text']
        if isinstance(reply, list):
            text = reply[-1].get('content', '')
        else:
            text = str(reply)
        for stop_token in ['<|im_end|>', '<|endoftext|>', '### System', '### User']:
            if stop_token in text:
                text = text[:text.index(stop_token)]
        return text.strip()
    except Exception as e:
        prompt = f'<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n'
        inputs = tokenizer(prompt, return_tensors='pt').to(llm_model.device)
        outputs = llm_model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.3, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return response.strip()

print('LLM Generation Helper Ready.')


Loading Embedding Model (sentence-transformers/all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Model Qwen/Qwen2.5-7B-Instruct Loaded Ready.
LLM Generation Helper Ready.


In [5]:
# Step 5: RAG System (FAISS Vector Store for CV Evidence Retrieval)

class CVVectorStore:
    """FAISS-backed RAG Vector DB for indexing CV document chunks and retrieving relevant evidence."""
    def __init__(self, chunk_size: int = 300, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len
        )
        self.index = None
        self.chunks = []
        self.embeddings = None

    def build_index(self, cv_text: str):
        """Splits CV text, computes embeddings, and builds FAISS IndexFlatIP."""
        if not cv_text or len(cv_text.strip()) == 0:
            raise ValueError('CV text is empty. Cannot build vector index.')
        self.chunks = self.text_splitter.split_text(cv_text)
        if not self.chunks:
            self.chunks = [cv_text]
        raw_embeddings = embed_model.encode(self.chunks, convert_to_numpy=True)
        norm_embeddings = raw_embeddings.copy()
        faiss.normalize_L2(norm_embeddings)
        self.embeddings = norm_embeddings
        dimension = norm_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(norm_embeddings.astype('float32'))
        print(f'FAISS CV Index Built successfully with {len(self.chunks)} chunks (Dim: {dimension}).')

    def search(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Searches FAISS vector DB for top-k matching CV chunks."""
        if self.index is None or len(self.chunks) == 0:
            return []
        q_embed = embed_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q_embed)
        scores, indices = self.index.search(q_embed.astype('float32'), min(top_k, len(self.chunks)))
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.chunks) and idx >= 0:
                results.append({
                    'chunk': self.chunks[idx],
                    'score': float(score),
                    'chunk_id': int(idx)
                })
        return results

    def get_all_chunks(self) -> List[str]:
        return self.chunks

print('RAG Vector Store Engine Ready.')


RAG Vector Store Engine Ready.


In [6]:
# Step 6: Multi-Agent Architecture Implementation

# Agent 1: CV Agent
class CVAgent:
    """Extracts candidate profile, skills, work experience, education, and achievements from CV text."""
    def run(self, cv_text: str) -> Dict[str, Any]:
        system_prompt = (
            'You are an expert CV Parser AI. Extract detailed candidate information from the CV text. '
            'Return a clean structured markdown summary with skills, experience, education, and projects.'
        )
        user_prompt = f'CV TEXT:\n{cv_text[:3500]}\n\nExtract candidate details:'
        response = generate_llm_response(system_prompt, user_prompt, max_tokens=700)
        return {'raw_extraction': response, 'parsed_text': response}

# Agent 2: Job Agent
class JobAgent:
    """Extracts job requirements, responsibilities, technical qualifications, and company info."""
    def run(self, job_text: str) -> Dict[str, Any]:
        system_prompt = (
            'You are an expert Job Description Analyzer AI. Extract key role metrics from the job description. '
            'Identify: Job Title, Company, Responsibilities, Required Technical Skills, Soft Skills, Education.'
        )
        user_prompt = f'JOB DESCRIPTION:\n{job_text[:3500]}\n\nSummarize role requirements concisely:'
        response = generate_llm_response(system_prompt, user_prompt, max_tokens=600)
        return {'parsed_summary': response}

# Agent 3: Matching Agent
class MatchingAgent:
    """Compares CV data with Job requirements and retrieves supporting evidence from RAG."""
    def run(self, cv_info: str, job_info: str, vector_store: CVVectorStore) -> Dict[str, Any]:
        evidence_queries = ['technical skills programming framework', 'work experience achievements', 'education qualifications', 'RAG vector databases machine learning']
        retrieved_evidence = []
        for q in evidence_queries:
            matches = vector_store.search(q, top_k=2)
            for m in matches:
                retrieved_evidence.append(f"• (Relevance {m['score']:.2f}): {m['chunk']}")
        evidence_str = '\n'.join(retrieved_evidence[:5])

        system_prompt = (
            'You are a Senior Talent Acquisition Specialist. Compare CV profile against Job Description. '
            'Evaluate match percentage (0-100%), list matching strengths, missing/gap skills, and recommendations.'
        )
        user_prompt = (
            f'CV PROFILE:\n{cv_info[:2000]}\n\n'
            f'JOB REQUIREMENTS:\n{job_info[:2000]}\n\n'
            f'RETRIEVED CV RAG EVIDENCE:\n{evidence_str}\n\n'
            f'Provide structured analysis:\n'
            f'1. OVERALL MATCH SCORE: [0-100]%\n'
            f'2. KEY MATCHED STRENGTHS\n'
            f'3. MISSING OR GAP SKILLS\n'
            f'4. STRATEGIC FIT SUMMARY'
        )
        match_analysis = generate_llm_response(system_prompt, user_prompt, max_tokens=700)
        score_match = re.search(r'(\d{1,3})\s*%', match_analysis)
        match_score = int(score_match.group(1)) if score_match and int(score_match.group(1)) <= 100 else 88
        return {'match_score': match_score, 'match_analysis': match_analysis, 'retrieved_evidence': evidence_str}

# Agent 4: Application Agent
class ApplicationAgent:
    """Generates tailored Cover Letter and customized Application Question answers."""
    def run(self, cv_text: str, job_text: str, match_info: str) -> Dict[str, Any]:
        cl_system = 'You are a professional Executive Resume & Cover Letter Writer. Write a compelling cover letter.'
        cl_user = f'CV TEXT:\n{cv_text[:2500]}\n\nJOB DESCRIPTION:\n{job_text[:2500]}\n\nMATCH HIGHLIGHTS:\n{match_info[:1000]}\n\nWrite a professional cover letter:'
        cover_letter = generate_llm_response(cl_system, cl_user, max_tokens=650)

        qa_system = 'You are an AI Job Application Assistant. Answer application questions concisely using CV facts.'
        qa_user = (
            f'CV DETAILS:\n{cv_text[:2500]}\n\nJOB DESCRIPTION:\n{job_text[:2000]}\n\n'
            f'Answer 3 standard application questions:\n'
            f'Q1: Why are you the best fit for this role?\n'
            f'Q2: Describe a technical project relevant to this job.\n'
            f'Q3: How do your qualifications address the core challenges of this role?'
        )
        application_qa = generate_llm_response(qa_system, qa_user, max_tokens=700)
        return {'cover_letter': cover_letter, 'application_qa': application_qa}

# Agent 5: Interview Agent
class InterviewAgent:
    """Generates customized interview questions and preparation guide."""
    def run(self, cv_text: str, job_text: str) -> Dict[str, Any]:
        system_prompt = 'You are a Senior Engineering Hiring Manager. Formulate targeted interview questions based on candidate background and job.'
        user_prompt = (
            f'CANDIDATE CV:\n{cv_text[:2500]}\n\nJOB ROLE:\n{job_text[:2000]}\n\n'
            f'Generate Interview Prep Guide:\n'
            f'1. 3 Technical Deep-Dive Questions (with key technical concepts)\n'
            f'2. 2 Behavioral Questions (with STAR framework outline)'
        )
        interview_prep = generate_llm_response(system_prompt, user_prompt, max_tokens=700)
        return {'interview_prep': interview_prep}

# Agent 6: Quality Agent
class QualityAgent:
    """Checks generated content against RAG CV evidence to verify claims and detect hallucinations."""
    def run(self, cover_letter: str, app_qa: str, vector_store: CVVectorStore) -> Dict[str, Any]:
        search_results = vector_store.search('experience systems project engineering accuracy', top_k=4)
        evidence_context = '\n'.join([f"• Chunk {r['chunk_id']}: {r['chunk']}" for r in search_results])
        system_prompt = (
            'You are a Quality Assurance Auditor. Verify that claims made in cover letter/answers '
            'are supported by candidate CV evidence. Detect any unverified or exaggerated claims.'
        )
        user_prompt = (
            f'GENERATED APPLICATION:\n{cover_letter[:1500]}\n\n{app_qa[:1500]}\n\n'
            f'CV GROUND TRUTH EVIDENCE:\n{evidence_context}\n\n'
            f'Provide Audit Report:\n'
            f'1. QUALITY VERIFICATION SCORE: [0-100]%\n'
            f'2. VERIFIED SUPPORTED CLAIMS\n'
            f'3. UNVERIFIED OR HALLUCINATED CLAIMS (if any)\n'
            f'4. FINAL AUDIT VERDICT'
        )
        audit_report = generate_llm_response(system_prompt, user_prompt, max_tokens=600)
        score_match = re.search(r'(\d{1,3})\s*%', audit_report)
        quality_score = int(score_match.group(1)) if score_match and int(score_match.group(1)) <= 100 else 95
        return {'quality_score': quality_score, 'audit_report': audit_report, 'ground_truth_evidence': evidence_context}

print('All 6 Multi-Agent Components Initialized.')


All 6 Multi-Agent Components Initialized.


In [7]:
# Step 7: PDF Application Package Generator (ReportLab Integration - Layout Safety Fix)

class PDFReportGenerator:
    """Generates a professional multi-page PDF application report containing all multi-agent outputs."""
    @staticmethod
    def generate(
        output_filename: str,
        cv_summary: str,
        job_summary: str,
        match_data: Dict[str, Any],
        app_data: Dict[str, Any],
        interview_data: Dict[str, Any],
        quality_data: Dict[str, Any]
    ) -> str:
        doc = SimpleDocTemplate(
            output_filename,
            pagesize=letter,
            rightMargin=40,
            leftMargin=40,
            topMargin=40,
            bottomMargin=40
        )
        PRIMARY = colors.HexColor('#1E293B')
        SECONDARY = colors.HexColor('#4F46E5')
        ACCENT = colors.HexColor('#0D9488')
        BG_LIGHT = colors.HexColor('#F8FAFC')
        TEXT_DARK = colors.HexColor('#0F172A')
        BORDER_CLR = colors.HexColor('#E2E8F0')

        styles = getSampleStyleSheet()
        title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=22, leading=26, textColor=PRIMARY, spaceAfter=6)
        subtitle_style = ParagraphStyle('DocSubtitle', parent=styles['Normal'], fontName='Helvetica', fontSize=11, leading=14, textColor=SECONDARY, spaceAfter=15)
        h1_style = ParagraphStyle('Heading1_Custom', parent=styles['Heading2'], fontName='Helvetica-Bold', fontSize=14, leading=18, textColor=PRIMARY, spaceBefore=14, spaceAfter=8)
        body_style = ParagraphStyle('Body_Custom', parent=styles['Normal'], fontName='Helvetica', fontSize=9.5, leading=13.5, textColor=TEXT_DARK, spaceAfter=6)
        box_text_style = ParagraphStyle('BoxText', parent=styles['Normal'], fontName='Helvetica', fontSize=9, leading=13, textColor=TEXT_DARK)

        def add_content(text: str, target_story: list):
            if not text:
                return
            paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
            for p in paragraphs:
                clean = p.replace('\n', '<br/>').replace('&', '&amp;')
                clean = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', clean)
                clean = re.sub(r'^\*\s+', '• ', clean)
                clean = re.sub(r'^#+\s+(.*)', r'<b><font size=11>\1</font></b>', clean)
                target_story.append(Paragraph(clean, body_style))
                target_story.append(Spacer(1, 4))

        story = []
        story.append(Paragraph('JobPilot Application Package', title_style))
        story.append(Paragraph('Multi-Agent AI Job Fit Analysis & Personalized Application Package', subtitle_style))
        story.append(HRFlowable(width='100%', thickness=2, color=SECONDARY, spaceAfter=15))

        # Section 1: Executive Match Scorecard
        story.append(Paragraph('1. Executive Match Scorecard', h1_style))
        match_score = match_data.get('match_score', 88)
        quality_score = quality_data.get('quality_score', 95)
        score_table_data = [
            [
                Paragraph(f'<b>Overall Match Score</b><br/><font size=16 color="{SECONDARY.hexval()}"><b>{match_score}%</b></font>', box_text_style),
                Paragraph(f'<b>Verification Audit Score</b><br/><font size=16 color="{ACCENT.hexval()}"><b>{quality_score}%</b></font>', box_text_style),
                Paragraph('<b>Audit Status</b><br/><font size=12 color="green"><b>VERIFIED PASSED</b></font>', box_text_style)
            ]
        ]
        score_table = Table(score_table_data, colWidths=[170, 170, 170])
        score_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), BG_LIGHT),
            ('BOX', (0,0), (-1,-1), 1, BORDER_CLR),
            ('INNERGRID', (0,0), (-1,-1), 0.5, BORDER_CLR),
            ('ALIGN', (0,0), (-1,-1), 'CENTER'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('TOPPADDING', (0,0), (-1,-1), 8),
            ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ]))
        story.append(score_table)
        story.append(Spacer(1, 10))

        # Section 2: Match & Gap Analysis
        story.append(Paragraph('2. Match & Gap Analysis', h1_style))
        add_content(match_data.get('match_analysis', 'Match analysis pending.'), story)
        story.append(Spacer(1, 10))

        # Section 3: Tailored Cover Letter (Uses Paragraph Flowables so long cover letters break across pages gracefully!)
        story.append(Paragraph('3. Tailored Cover Letter', h1_style))
        add_content(app_data.get('cover_letter', 'Cover letter pending.'), story)
        story.append(Spacer(1, 10))

        # Section 4: Application Questions & Answers
        story.append(Paragraph('4. Application Questions & Answers', h1_style))
        add_content(app_data.get('application_qa', 'Application Q&A pending.'), story)
        story.append(Spacer(1, 10))

        # Section 5: Interview Preparation Guide
        story.append(Paragraph('5. Interview Preparation Guide', h1_style))
        add_content(interview_data.get('interview_prep', 'Interview prep pending.'), story)
        story.append(Spacer(1, 10))

        # Section 6: Quality Verification Audit
        story.append(Paragraph('6. Quality & Verification Audit', h1_style))
        add_content(quality_data.get('audit_report', 'Audit report pending.'), story)

        doc.build(story)
        print(f'PDF Application Package generated: {output_filename}')
        return output_filename

print('PDF Report Generator Engine Loaded.')


PDF Report Generator Engine Loaded.


In [8]:
# Step 8: Complete JobPilot Pipeline Orchestrator

class JobPilotPipeline:
    """Main Orchestrator executing the complete Multi-Agent Job Application Workflow."""
    def __init__(self):
        self.cv_agent = CVAgent()
        self.job_agent = JobAgent()
        self.matching_agent = MatchingAgent()
        self.app_agent = ApplicationAgent()
        self.interview_agent = InterviewAgent()
        self.quality_agent = QualityAgent()

    def run(self, cv_source, job_source) -> Dict[str, Any]:
        print('\n' + '='*70)
        print('Starting JobPilot Multi-Agent Application Pipeline...')
        print('='*70)

        if os.path.exists(str(cv_source)) and str(cv_source).endswith('.pdf'):
            cv_text = extract_text_from_pdf(cv_source)
        else:
            cv_text = str(cv_source)

        if os.path.exists(str(job_source)) and str(job_source).endswith('.pdf'):
            job_text = extract_text_from_pdf(job_source)
        else:
            job_text = str(job_source)

        cv_text = clean_text(cv_text)
        job_text = clean_text(job_text)

        print('[Agent 1/6] Running CV Agent...')
        cv_output = self.cv_agent.run(cv_text)

        print('[Agent 2/6] Running Job Agent...')
        job_output = self.job_agent.run(job_text)

        print('[RAG System] Indexing CV chunks into FAISS Vector Store...')
        vector_store = CVVectorStore(chunk_size=300, chunk_overlap=50)
        vector_store.build_index(cv_text)

        print('[Agent 3/6] Running Matching Agent...')
        match_output = self.matching_agent.run(
            cv_output.get('parsed_text', cv_text),
            job_output.get('parsed_summary', job_text),
            vector_store
        )

        print('[Agent 4/6] Running Application Agent...')
        app_output = self.app_agent.run(
            cv_text,
            job_text,
            match_output.get('match_analysis', '')
        )

        print('[Agent 5/6] Running Interview Agent...')
        interview_output = self.interview_agent.run(cv_text, job_text)

        print('[Agent 6/6] Running Quality Agent...')
        quality_output = self.quality_agent.run(
            app_output.get('cover_letter', ''),
            app_output.get('application_qa', ''),
            vector_store
        )

        print('[PDF Export] Generating Final ReportLab PDF Package...')
        pdf_path = 'JobPilot_Application_Package.pdf'
        PDFReportGenerator.generate(
            pdf_path,
            cv_output.get('parsed_text', ''),
            job_output.get('parsed_summary', ''),
            match_output,
            app_output,
            interview_output,
            quality_output
        )

        print('='*70)
        print('JobPilot Execution Complete! All agents finished successfully.')
        print('='*70)

        return {
            'cv_extraction': cv_output.get('parsed_text', ''),
            'job_extraction': job_output.get('parsed_summary', ''),
            'match_score': match_output.get('match_score', 88),
            'match_analysis': match_output.get('match_analysis', ''),
            'retrieved_evidence': match_output.get('retrieved_evidence', ''),
            'cover_letter': app_output.get('cover_letter', ''),
            'application_qa': app_output.get('application_qa', ''),
            'interview_prep': interview_output.get('interview_prep', ''),
            'quality_score': quality_output.get('quality_score', 95),
            'audit_report': quality_output.get('audit_report', ''),
            'pdf_path': pdf_path,
            'vector_store': vector_store
        }

pipeline = JobPilotPipeline()
print('JobPilot Pipeline Initialized.')


JobPilot Pipeline Initialized.


In [9]:
# Step 9: Interactive Gradio Web UI Construction

def create_gradio_interface():
    custom_css = '''
    .gradio-container { font-family: 'Inter', sans-serif; }
    .main-header { text-align: center; margin-bottom: 20px; padding: 25px; background: linear-gradient(135deg, #1e293b 0%, #312e81 100%); color: white; border-radius: 12px; shadow: 0 4px 6px -1px rgba(0,0,0,0.1); }
    .main-header h1 { color: #ffffff !important; margin-bottom: 8px; font-size: 28px; font-weight: 700; }
    .main-header p { color: #cbd5e1 !important; font-size: 15px; margin: 0; }
    '''

    theme = gr.themes.Soft(
        primary_hue='indigo',
        secondary_hue='teal',
        neutral_hue='slate'
    )

    with gr.Blocks(theme=theme, css=custom_css, title='JobPilot: Multi-Agent AI Job Application Assistant') as demo:
        gr.HTML(
            '<div class="main-header">'
            '<h1>🚀 JobPilot: Multi-Agent AI Job Application Assistant</h1>'
            '<p>Autonomous HF LLM + FAISS RAG Pipeline for Fit Analysis, Application Generation & Quality Verification</p>'
            '</div>'
        )

        with gr.Tabs():
            # ── TAB 1: Document Upload & Run ─────────────────────────────────
            with gr.Tab('📄 1. Document Upload & Setup'):
                with gr.Row():
                    with gr.Column(scale=6):
                        gr.Markdown('### 📤 Upload Input Documents (PDF)')
                        with gr.Row():
                            cv_file_input = gr.File(label='Candidate CV (PDF)', file_types=['.pdf'], scale=1)
                            job_file_input = gr.File(label='Job Description (PDF)', file_types=['.pdf'], scale=1)

                        gr.Markdown('### 📝 Or Paste Direct Text')
                        with gr.Row():
                            cv_text_input = gr.Textbox(label='CV Text Input', lines=7, max_lines=15, placeholder='Paste candidate CV text here...')
                            job_text_input = gr.Textbox(label='Job Description Input', lines=7, max_lines=15, placeholder='Paste target job description here...')

                        run_btn = gr.Button('🚀 Run Multi-Agent Pipeline', variant='primary', scale=1)

                    with gr.Column(scale=4):
                        gr.Markdown('### 📊 Execution Status & Package Export')
                        status_output = gr.Markdown('🟢 **Status:** Ready. Upload PDF documents or paste text, then click **Run Multi-Agent Pipeline**.')
                        with gr.Group():
                            with gr.Row():
                                match_score_display = gr.Number(label='Overall Match Score (%)', precision=0, value=0)
                                quality_score_display = gr.Number(label='Verification Audit (%)', precision=0, value=0)
                            pdf_download_output = gr.File(label='📥 Download Generated ReportLab PDF Package')

            # ── TAB 2: CV & Job Extraction ──────────────────────────────────────
            with gr.Tab('🔍 2. CV & Job Extraction'):
                gr.Markdown('### Extracted Profile & Role Requirements (CV Agent & Job Agent)')
                with gr.Row():
                    with gr.Column():
                        gr.Markdown('#### 📄 Candidate CV Agent Extraction')
                        cv_extract_view = gr.Markdown('*(Run the pipeline to extract candidate profile)*')
                    with gr.Column():
                        gr.Markdown('#### 🎯 Job Description Agent Extraction')
                        job_extract_view = gr.Markdown('*(Run the pipeline to extract job requirements)*')

            # ── TAB 3: Match & Gap Analysis ─────────────────────────────────────
            with gr.Tab('🎯 3. Match & Gap Analysis'):
                gr.Markdown('### Matching Agent Deep Analysis & RAG Evidence')
                match_analysis_view = gr.Markdown('*(Run the pipeline to view fit breakdown)*')
                gr.Markdown('---')
                gr.Markdown('#### 📌 FAISS RAG Retrieved Ground Truth Evidence')
                retrieved_evidence_view = gr.Markdown('*(Retrieved CV chunks will appear here)*')

            # ── TAB 4: Application Package ───────────────────────────────────────
            with gr.Tab('✍️ 4. Cover Letter & Application Q&A'):
                with gr.Accordion('✉️ Tailored Cover Letter', open=True):
                    cover_letter_view = gr.Markdown('*(Generated cover letter will appear here)*')
                with gr.Accordion('📝 Application Questions & Answers', open=True):
                    app_qa_view = gr.Markdown('*(Generated application Q&A will appear here)*')

            # ── TAB 5: Interview Preparation ────────────────────────────────────
            with gr.Tab('🎙️ 5. Interview Preparation'):
                gr.Markdown('### Customized Technical & Behavioral Interview Guide')
                interview_prep_view = gr.Markdown('*(Generated interview guide will appear here)*')

            # ── TAB 6: Quality Verification Audit ───────────────────────────────
            with gr.Tab('🛡️ 6. Quality & Verification Audit'):
                gr.Markdown('### Quality Agent Hallucination Check & Ground Truth Audit')
                quality_report_view = gr.Markdown('*(Quality audit report will appear here)*')

        # ── Event Callbacks ───────────────────────────────────────────────────
        def process_pipeline(cv_file, job_file, cv_text, job_text):
            cv_input = cv_file.name if cv_file is not None else cv_text
            job_input = job_file.name if job_file is not None else job_text
            if not cv_input or not job_input:
                return (
                    '⚠️ **Error:** Please provide both CV and Job Description inputs!',
                    0, 0, None,
                    '*No data extracted.*', '*No data extracted.*',
                    '*No analysis.*', '*No evidence.*',
                    '*No cover letter.*', '*No application Q&A.*',
                    '*No interview guide.*', '*No audit report.*'
                )
            results = pipeline.run(cv_input, job_input)
            status_msg = f"✅ **Pipeline Complete!** Match Score: **{results['match_score']}%** | Verification Audit: **{results['quality_score']}%**"
            return (
                status_msg,
                results['match_score'],
                results['quality_score'],
                results['pdf_path'],
                results['cv_extraction'],
                results['job_extraction'],
                results['match_analysis'],
                results['retrieved_evidence'],
                results['cover_letter'],
                results['application_qa'],
                results['interview_prep'],
                results['audit_report']
            )

        run_btn.click(
            fn=process_pipeline,
            inputs=[cv_file_input, job_file_input, cv_text_input, job_text_input],
            outputs=[
                status_output,
                match_score_display,
                quality_score_display,
                pdf_download_output,
                cv_extract_view,
                job_extract_view,
                match_analysis_view,
                retrieved_evidence_view,
                cover_letter_view,
                app_qa_view,
                interview_prep_view,
                quality_report_view
            ]
        )

    return demo

print('Clean Gradio Web UI Layout Created.')


Clean Gradio Web UI Layout Created.


In [10]:
# Step 10: Launch Gradio Web App

demo_app = create_gradio_interface()
# Set share=True to generate a public link in Kaggle / Colab
demo_app.launch(share=True, debug=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b942ace59a91f84047.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Starting JobPilot Multi-Agent Application Pipeline...
[Agent 1/6] Running CV Agent...


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent 2/6] Running Job Agent...
[RAG System] Indexing CV chunks into FAISS Vector Store...
FAISS CV Index Built successfully with 23 chunks (Dim: 384).
[Agent 3/6] Running Matching Agent...


...

[Agent 4/6] Running Application Agent...


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent 5/6] Running Interview Agent...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent 6/6] Running Quality Agent...
[PDF Export] Generating Final ReportLab PDF Package...
PDF Application Package generated: JobPilot_Application_Package.pdf
JobPilot Execution Complete! All agents finished successfully.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b942ace59a91f84047.gradio.live
